<a href="https://colab.research.google.com/github/LaurenMitchell-tech/uvvisml/blob/main/notebooks/Multitask_Multicomponent_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Run to allow gpu use. Skip if using cpu. Usually takes a few minutes.
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install .
        %cd examples

from lightning import pytorch as pl
import torch
import numpy as np
import pandas as pd
from pathlib import Path

import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

from chemprop import data, featurizers, models, nn
from chemprop.nn import metrics
from chemprop.models import multi

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
chemprop_dir = Path.cwd().parent
input_path = chemprop_dir / "examples" / "all_optical_data_including_duplicates.csv"
smiles_columns = ['smiles', 'solvent']
target_columns = ['absorption_max','emission_max']

df_input = pd.read_csv(input_path, usecols=['smiles', 'solvent', 'absorption_max','emission_max'])
df_input.dropna(inplace=True)
smiss = df_input.loc[:, smiles_columns].values
ys = df_input.loc[:, target_columns].values

datapoints = [[data.MoleculeDatapoint.from_smi(smis[0], y) for smis, y in zip(smiss, ys)]]
datapoints += [[data.MoleculeDatapoint.from_smi(smis[i]) for smis in smiss] for i in range(1, len(smiles_columns))]

In [ ]:
df_input

In [ ]:
#multicomponent code
component_to_split_by = 0 # index of the component to use for structure based splits
mols = [d.mol for d in datapoints[component_to_split_by]]

train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.8, 0.1, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    datapoints, train_indices, val_indices, test_indices
)

In [ ]:
#Display test set
test_indices = [idx for sublist in test_indices for idx in sublist]
df_test = df_input.iloc[test_indices].reset_index(drop=True)
df_test

In [ ]:
#multicomponent code
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_dset = [data.MoleculeDataset(train_data[0][i], featurizer) for i in range(len(smiles_columns))]
val_dset = [data.MoleculeDataset(val_data[0][i], featurizer) for i in range(len(smiles_columns))]
test_dset = [data.MoleculeDataset(test_data[0][i], featurizer) for i in range(len(smiles_columns))]

train_mcdset = data.MulticomponentDataset(train_dset)
scaler = train_mcdset.normalize_targets()
val_mcdset = data.MulticomponentDataset(val_dset)
val_mcdset.normalize_targets(scaler)
test_mcdset = data.MulticomponentDataset(test_dset)

BATCH_SIZE = 256
train_loader = data.build_dataloader(train_mcdset, batch_size=BATCH_SIZE)
val_loader = data.build_dataloader(val_mcdset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = data.build_dataloader(test_mcdset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
#multicomponent code

torch.manual_seed(0) #use same random seed every time
np.random.seed(0)

mcmp = nn.MulticomponentMessagePassing(
    blocks=[nn.BondMessagePassing() for _ in range(len(smiles_columns))],
    n_components=len(smiles_columns),
)

agg = nn.MeanAggregation()

In [ ]:
#multicomponent code, could add dropout in the FFN (e.g., dropout=0.2–0.5)
output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)

ffn = nn.RegressionFFN(
    n_tasks = len(target_columns),
    input_dim=mcmp.output_dim,
    output_transform=output_transform,
)

metric_list = [metrics.RMSE(), metrics.MAE()] # Only the first metric is used for training and early stopping

In [ ]:
#multicomponent code
mcmpnn = multi.MulticomponentMPNN(
    mcmp,
    agg,
    ffn,
    metrics=metric_list,
)

mcmpnn

In [ ]:
#code for training a model which will be saved
checkpoint = pl.callbacks.ModelCheckpoint(
    dirpath="checkpoints/",
    monitor="val_loss",
    save_top_k=1,
    mode="min",
    filename="all_data_abs_emi"
)

early_stop = pl.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    mode="min"
)

from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = torch.optim.AdamW(mcmpnn.parameters(), lr=2e-4, weight_decay=1e-6)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [ ]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

In [ ]:
#multicomponent code
trainer = pl.Trainer(accelerator="auto", logger=True, callbacks=[checkpoint, early_stop], max_epochs=100, deterministic=True) #enable_checkpointing=False,

In [ ]:
#multicomponent code
trainer.fit(mcmpnn, train_loader, val_loader)

In [ ]:
results = trainer.test(mcmpnn, test_loader, weights_only=False)  # weights_only=False is only required with pytorch lightning version 2.6.0 or newer

In [ ]:
#Show where model is saved
best_model_path = checkpoint.best_model_path
print(f"Best model saved at: {best_model_path}")

In [ ]:
#Code for loading different model than the one just trained
path = chemprop_dir / "examples" / "CIE_model_nm.ckpt"
mcmpnn = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu", weights_only=False)

In [ ]:
#Predict and show results compared to known data
preds = trainer.predict(mcmpnn, test_loader) #Works with mcmpnn or chemprop_model
preds_tensor = torch.cat(preds, dim=0)

preds_array = preds_tensor.detach().cpu().numpy()

columns = ['abs_pred', 'emi_pred']
df_test[columns] = preds_array

df_test

In [ ]:
def plot_and_save(true_values=str, preds=str, model=str, value_type=str, save_path=str):

  '''
  true_values: name of column containing experimental x or y values
  preds: name of column containing predicted x or y values
  model: name of model used to make prediction
  save_path: path where figure will be saved

  plots a parity plot and saves the figure to the specified path
  '''

  true_ys = df_test[true_values].tolist()
  preds_y = df_test[preds].tolist()

  fig = plt.figure(figsize=(8,5))
  plt.scatter(true_ys, preds_y, color='red', marker='o', label='Parity Plot')
  ax = plt.gca()
  lims = ax.get_xlim()
  ax.plot(lims, lims, color='black')
  plt.legend()
  plt.xlabel('Experimental')
  plt.ylabel('Predicted')
  title = 'Parity Plot of ' + model + ' ' + value_type + ' Values'
  plt.title(title)
  plt.grid(True)

  with open(save_path, "wb") as f:
      pickle.dump(fig, f)

  plt.show()

  print('Plot saved as: ' + save_path)

In [ ]:
plot_and_save('abs FWHM (cm-1)', 'abs_pred', 'Deep4Chem_abs_emi', 'abs', 'Abs_Parity_abs_emi_model.pkl')
files.download('Abs_Parity_abs_emi_model.pkl')

In [ ]:
plot_and_save('emi FWHM (cm-1)', 'emi_pred', 'Deep4Chem_abs_emi', 'emi', 'Emi_Parity_abs_emi_model.pkl')
files.download('Emi_Parity_abs_emi_model.pkl')